# Count low-weight undetectable errors

This notebook counts weight `d`, `d+1`, and `d+2` undetectable errors in two senses:

- classical undetectable errors: nonzero vectors in `ker(H)`
- CSS logical-basis representatives: combinations of the logical `X` and `Z` basis returned by `get_logical_operators_by_pivoting`

Set the HDF5 `filepath`, `code_name`, and `run_name` in the next cell before rerunning on a different best code.

In [1]:
import h5py
from optimization.analyze_codes.decoder_performance_from_state import evaluate_performance_of_state
from optimization.experiments_settings import from_edgelist

filepath = "optimization/results/start_from_logical_guided.hdf5"
code_name = "[1600,64]"  # Update this key if needed
p = 0.03
MC_budget = int(1e6)
run_name = "beam_from_logical_bw2_d10_p0.03_100000screen_1000000prec_exp50_20260412_132444"


with h5py.File(filepath, "a") as f:
    code_grp = f[code_name]
    run_grp = code_grp[run_name]

    # Load the best state found by the search run.
    best_state_edge_list = run_grp["best_state"][:]
    best_state = from_edgelist(best_state_edge_list)
    best_ler = run_grp.attrs["min_cost"]
    print(best_state_edge_list)
    print(best_ler)

    # Reuse the best state in the analysis cells below.
    state = best_state

[[ 0 24  0 32  0 39  0 49  1 24  1 33  1 40  1 48  1 55  2 32  2 34  2 50
   2 52  3 25  3 32  3 36  3 42  3 55  4 25  4 35  4 41  4 51  5 25  5 37
   5 43  5 50  6 24  6 26  6 44  6 51  7 37  7 38  7 40  7 41  8 26  8 33
   8 38  8 41  9 27  9 42  9 52  9 53 10 27 10 31 10 35 10 44 11 27 11 37
  11 43 11 49 12 28 12 33 12 45 12 51 13 28 13 36 13 44 13 50 14 28 14 38
  14 46 14 47 15 29 15 34 15 42 15 46 16 29 16 36 16 47 16 52 17 29 17 45
  17 48 17 54 18 30 18 34 18 43 18 54 19 26 19 30 19 49 19 53 20 30 20 39
  20 45 21 31 21 35 21 46 21 54 22 31 22 39 22 48 22 55 23 40 23 47 23 53]]
0.001276


In [2]:
import numpy as np
import ldpc
import ldpc.code_util
from scipy.sparse import csr_matrix

from basic_css_code import construct_HGP_code
from optimization.experiments_settings import tanner_graph_to_parity_check_matrix
from optimization.undetectable_errors import (
    count_classical_undetectable_errors,
    count_css_logical_basis_representatives,
    sample_weights_undetectable_errors,
)

# None means exact over the full span. Use a small integer for a bounded search.
CLASSICAL_MAX_COMB_ORDER = None

# Exact CSS enumeration for this file would require 2^25 - 1 logical-basis combinations per side.
# Keep this bounded unless you specifically want the full exponential run.
CSS_MAX_COMB_ORDER = 5

In [3]:
H = tanner_graph_to_parity_check_matrix(state)
csr_H = csr_matrix(H, dtype=np.uint8)

n_classical, k_classical, _ = ldpc.code_util.compute_code_parameters(csr_H)
d_classical = ldpc.code_util.compute_exact_code_distance(csr_H)
rank_H = ldpc.mod2.rank(csr_H)

if k_classical == n_classical - csr_H.shape[0]:
    n_T_classical = csr_H.shape[0]
    k_T_classical = n_T_classical - rank_H
    d_T_classical = np.inf
else:
    n_T_classical, k_T_classical, d_T_classical = ldpc.code_util.compute_code_parameters(
        csr_matrix(H.T, dtype=np.uint8)
    )

d_quantum = min(d_classical, d_T_classical)
Hx, Hz = construct_HGP_code(csr_H)

print("Loaded best state from the HDF5 run.")
print(f"H shape: {csr_H.shape}")
print(f"Classical H: [{n_classical}, {k_classical}, {d_classical}]")
print(f"Classical H.T: [{n_T_classical}, {k_T_classical}, {d_T_classical}]")
print(f"HGP CSS matrices: Hx={Hx.shape}, Hz={Hz.shape}")
print(f"Quantum distance used for target weights: {d_quantum}")

Loaded best state from the HDF5 run.
H shape: (24, 32)
Classical H: [32, 8, 12]
Classical H.T: [24, 0, inf]
HGP CSS matrices: Hx=(768, 1600), Hz=(768, 1600)
Quantum distance used for target weights: 12


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ldpc/code_util/code_util.py:164: UserWarning: This function has exponential complexity. Not recommended for large pcms. Use the                            'ldpc.code_util.estimate_code_distance' function instead.
  warnings.warn(


In [4]:
def summarize_regularity(matrix, name):
    row_w = np.asarray(matrix.sum(axis=1)).ravel().astype(int)
    col_w = np.asarray(matrix.sum(axis=0)).ravel().astype(int)

    row_unique = np.unique(row_w)
    col_unique = np.unique(col_w)

    print(f"{name}:")
    print(f"  shape: {matrix.shape}")
    print(f"  row weights: min={row_w.min()}, max={row_w.max()}, unique={row_unique.tolist()}")
    print(f"  col weights: min={col_w.min()}, max={col_w.max()}, unique={col_unique.tolist()}")
    print(f"  row-regular: {len(row_unique) == 1}")
    print(f"  col-regular: {len(col_unique) == 1}")
    print(f"  fully regular: {len(row_unique) == 1 and len(col_unique) == 1}")
    print()

print("Regularity summary for the current code:")
summarize_regularity(csr_H, "Classical H")
summarize_regularity(Hx, "Hx")
summarize_regularity(Hz, "Hz")

Regularity summary for the current code:
Classical H:
  shape: (24, 32)
  row weights: min=3, max=5, unique=[3, 4, 5]
  col weights: min=3, max=3, unique=[3]
  row-regular: False
  col-regular: True
  fully regular: False

Hx:
  shape: (768, 1600)
  row weights: min=6, max=8, unique=[6, 7, 8]
  col weights: min=3, max=5, unique=[3, 4, 5]
  row-regular: False
  col-regular: False
  fully regular: False

Hz:
  shape: (768, 1600)
  row weights: min=6, max=8, unique=[6, 7, 8]
  col weights: min=3, max=5, unique=[3, 4, 5]
  row-regular: False
  col-regular: False
  fully regular: False



In [5]:
def is_circulant_block(block: np.ndarray) -> bool:
    if block.shape[0] != block.shape[1]:
        return False
    first_row = block[0]
    return all(np.array_equal(block[i], np.roll(first_row, i)) for i in range(block.shape[0]))


def summarize_block_structure(matrix, name):
    rows, cols = matrix.shape
    candidates = [d for d in range(2, min(rows, cols) + 1) if rows % d == 0 and cols % d == 0]

    print(f"{name} block-structure scan:")
    if not candidates:
        print("  No common nontrivial block sizes divide both dimensions.")
        print()
        return

    for lift in candidates:
        block_rows = rows // lift
        block_cols = cols // lift
        unique_blocks = {}
        total_blocks = block_rows * block_cols
        circulant_blocks = 0

        for br in range(block_rows):
            for bc in range(block_cols):
                block = np.asarray(matrix[br * lift:(br + 1) * lift, bc * lift:(bc + 1) * lift], dtype=np.uint8)
                sig = tuple(block.ravel().tolist())
                unique_blocks[sig] = unique_blocks.get(sig, 0) + 1
                if is_circulant_block(block):
                    circulant_blocks += 1

        print(
            f"  lift={lift}: grid={block_rows}x{block_cols}, "
            f"unique_blocks={len(unique_blocks)}, "
            f"circulant_blocks={circulant_blocks}/{total_blocks}"
        )
    print()

print("Quasi-cyclic / block-structure scan for the current code:")
summarize_block_structure(csr_H.toarray(), "Classical H")
summarize_block_structure(Hx.toarray(), "Hx")
summarize_block_structure(Hz.toarray(), "Hz")

Quasi-cyclic / block-structure scan for the current code:
Classical H block-structure scan:
  lift=2: grid=12x16, unique_blocks=11, circulant_blocks=123/192
  lift=4: grid=6x8, unique_blocks=37, circulant_blocks=7/48
  lift=8: grid=3x4, unique_blocks=12, circulant_blocks=0/12

Hx block-structure scan:


KeyboardInterrupt: 

## Translation-equivalence and Tanner graph drawings

The next cell checks how many distinct local neighborhood patterns appear in the Tanner graph, which is a practical proxy for translation-equivalent structure. The following cell saves interactive Tanner graph visualizations for the classical code and the quantum HGP code.

In [6]:
from collections import Counter

from optimization.analyze_codes.draw_tanner_graph import draw_interactive_tanner_graph
from optimization.experiments_settings import from_parity_check_matrix


def node_neighborhood_signature(G, node):
    neighbor_degrees = sorted(G.degree(nbr) for nbr in G.neighbors(node))
    return tuple(neighbor_degrees)


def summarize_translation_equivariance(G, name):
    variable_nodes = [n for n, b in G.nodes(data="bipartite") if b == 1]
    check_nodes = [n for n, b in G.nodes(data="bipartite") if b == 0]

    variable_signatures = [node_neighborhood_signature(G, n) for n in variable_nodes]
    check_signatures = [node_neighborhood_signature(G, n) for n in check_nodes]

    variable_counts = Counter(variable_signatures)
    check_counts = Counter(check_signatures)

    print(f"{name} translation-equivalence proxy:")
    print(f"  variable nodes: {len(variable_nodes)}")
    print(f"  check nodes: {len(check_nodes)}")
    print(f"  distinct variable neighborhood signatures: {len(variable_counts)}")
    print(f"  distinct check neighborhood signatures: {len(check_counts)}")
    if variable_counts:
        sig, count = variable_counts.most_common(1)[0]
        print(f"  most common variable signature count: {count}")
        print(f"  most common variable signature: {sig}")
    if check_counts:
        sig, count = check_counts.most_common(1)[0]
        print(f"  most common check signature count: {count}")
        print(f"  most common check signature: {sig}")
    print()


print("Translation-equivalence proxy for the current code:")
summarize_translation_equivariance(state, "Classical Tanner graph")
summarize_translation_equivariance(from_parity_check_matrix(Hx), "Quantum Tanner graph from Hx")
summarize_translation_equivariance(from_parity_check_matrix(Hz), "Quantum Tanner graph from Hz")

# Draw interactive Tanner graphs. The quantum graphs are large, so these HTML files can be heavy.
classical_graph_file = "figures/best_code_classical_tanner_graph.html"
quantum_hx_graph_file = "figures/best_code_quantum_hx_tanner_graph.html"
quantum_hz_graph_file = "figures/best_code_quantum_hz_tanner_graph.html"

print("Saving Tanner graph visualizations...")
draw_interactive_tanner_graph(state, filename=classical_graph_file)
draw_interactive_tanner_graph(from_parity_check_matrix(Hx), filename=quantum_hx_graph_file)
draw_interactive_tanner_graph(from_parity_check_matrix(Hz), filename=quantum_hz_graph_file)
print("Saved:")
print(f"  {classical_graph_file}")
print(f"  {quantum_hx_graph_file}")
print(f"  {quantum_hz_graph_file}")

Translation-equivalence proxy for the current code:
Classical Tanner graph translation-equivalence proxy:
  variable nodes: 32
  check nodes: 24
  distinct variable neighborhood signatures: 5
  distinct check neighborhood signatures: 3
  most common variable signature count: 18
  most common variable signature: (4, 4, 4)
  most common check signature count: 20
  most common check signature: (3, 3, 3, 3)

Quantum Tanner graph from Hx translation-equivalence proxy:
  variable nodes: 1600
  check nodes: 768
  distinct variable neighborhood signatures: 13
  distinct check neighborhood signatures: 15
  most common variable signature count: 616
  most common variable signature: (7, 7, 7)
  most common check signature count: 360
  most common check signature: (3, 3, 3, 3, 4, 4, 4)

Quantum Tanner graph from Hz translation-equivalence proxy:
  variable nodes: 1600
  check nodes: 768
  distinct variable neighborhood signatures: 13
  distinct check neighborhood signatures: 15
  most common varia

In [ ]:
import h5py
from optimization.analyze_codes.decoder_performance_from_state import evaluate_performance_of_state
from optimization.experiments_settings import from_edgelist

filepath = "optimization/results/start_from_logical_guided.hdf5"
code_name = "[1600,64]"  # Update this key if needed
p = 0.03
MC_budget = int(1e6)
run_name = "beam_from_logical_bw2_d10_p0.03_100000screen_1000000prec_exp50_20260412_132444"


with h5py.File(filepath, "a") as f:
    code_grp = f[code_name]
    run_grp = code_grp[run_name]

    # Load the best state found by the search run.
    best_state_edge_list = run_grp["best_state"][:]
    best_state = from_edgelist(best_state_edge_list)
    best_ler = run_grp.attrs["min_cost"]
    print(best_state_edge_list)
    print(best_ler)

    # Reuse the best state in the analysis cells below.
    state = best_state

In [16]:
def get_tanner_graph_from_file(filepath, code_name, run_name):
    with h5py.File(filepath, "r") as f:
        code_grp = f[code_name]
        run_grp = code_grp[run_name]
        edge_list = run_grp["best_state"][:]
        return from_edgelist(edge_list)

In [21]:
def count_signature_equivalence_classes(state):
    # Compute and display distinct variable-neighborhood signatures (exact check indices)
    check_nodes = sorted([n for n, b in state.nodes(data='bipartite') if b == 0])
    var_nodes = sorted([n for n, b in state.nodes(data='bipartite') if b == 1])
    check_index = {n: i for i, n in enumerate(check_nodes)}
    var_index = {n: i for i, n in enumerate(var_nodes)}

    variable_patterns = {}
    for v in var_nodes:
        nbrs = sorted([check_index[n] for n in state.neighbors(v)])
        sig = tuple(nbrs)
        variable_patterns.setdefault(sig, []).append(v)

    # Sort patterns by occurrence
    sorted_variable_patterns = sorted(variable_patterns.items(), key=lambda kv: len(kv[1]), reverse=True)

    check_patterns = {}
    for c in check_nodes:
        nbrs = sorted([var_index[n] for n in state.neighbors(c)])
        sig = tuple(nbrs)
        check_patterns.setdefault(sig, []).append(c)
    
    # Sort patterns by occurrence
    sorted_check_patterns = sorted(check_patterns.items(), key=lambda kv: len(kv[1]), reverse=True)
    
    # Test cyclic-shift equivalence of variable-neighborhood signatures
    m = len(check_nodes)  # number of check nodes
    n = len(var_nodes)    # number of variable nodes

    def shift_sig(sig, s):
        return tuple(sorted(((i + s) % m) for i in sig))

    # Build canonical representative for each signature under cyclic shifts
    canonical_variable_map = {}
    for sig, vars_ in sorted_variable_patterns:
        reps = [shift_sig(sig, s) for s in range(m)]
        canonical = min(reps)
        canonical_variable_map.setdefault(canonical, []).append((sig, vars_))

    canonical_check_map = {}
    for sig, checks_ in sorted_check_patterns:
        reps = [shift_sig(sig, s) for s in range(n)]
        canonical = min(reps)
        canonical_check_map.setdefault(canonical, []).append((sig, checks_))

    print(f"Total variable cyclic-equivalence classes: {len(canonical_variable_map)}/{len(var_nodes)}")
    print(f"Total check cyclic-equivalence classes: {len(canonical_check_map)}/{len(check_nodes)}")

    print("\nCheck variable classes with cyclic-shift equivalence:")
    # Show classes that contain more than one distinct signature
    for i, (canon, members) in enumerate(sorted(canonical_variable_map.items(), key=lambda kv: -len(kv[1])), start=1):
        print(f"Class {i}: size={len(members)}")
        for sig, vars_ in members:
            print(f"  sig: {sig}  count: {len(vars_)}  vars: {[f'd{var_index[v]}' for v in vars_]}" )
        print()

    print("\nCheck node classes with cyclic-shift equivalence:")

    for i, (canon, members) in enumerate(sorted(canonical_check_map.items(), key=lambda kv: -len(kv[1])), start=1):
        print(f"Class {i}: size={len(members)}")
        for sig, checks_ in members:
            print(f"  sig: {sig}  count: {len(checks_)}  checks: {[f'c{check_index[c]}' for c in checks_]}")
        print()

In [22]:
import h5py
from optimization.analyze_codes.decoder_performance_from_state import evaluate_performance_of_state
from optimization.experiments_settings import from_edgelist

filepath = "optimization/results/start_from_logical_guided.hdf5"
code_name = "[1600,64]"  # Update this key if needed
p = 0.03
MC_budget = int(1e6)
run_name = "beam_from_logical_bw2_d10_p0.03_100000screen_1000000prec_exp50_20260412_132444"

state = get_tanner_graph_from_file(filepath, code_name, run_name)
count_signature_equivalence_classes(state)

Total variable cyclic-equivalence classes: 24/32
Total check cyclic-equivalence classes: 24/24

Check variable classes with cyclic-shift equivalence:
Class 1: size=5
  sig: (3, 4, 5)  count: 1  vars: ['d1']
  sig: (9, 10, 11)  count: 1  vars: ['d3']
  sig: (12, 13, 14)  count: 1  vars: ['d4']
  sig: (15, 16, 17)  count: 1  vars: ['d5']
  sig: (18, 19, 20)  count: 1  vars: ['d6']

Class 2: size=2
  sig: (2, 15, 18)  count: 1  vars: ['d10']
  sig: (2, 5, 13)  count: 1  vars: ['d26']

Class 3: size=2
  sig: (7, 8, 14)  count: 1  vars: ['d14']
  sig: (14, 15, 21)  count: 1  vars: ['d22']

Class 4: size=2
  sig: (1, 7, 23)  count: 1  vars: ['d16']
  sig: (4, 6, 12)  count: 1  vars: ['d27']

Class 5: size=2
  sig: (12, 17, 20)  count: 1  vars: ['d21']
  sig: (1, 17, 22)  count: 1  vars: ['d24']

Class 6: size=1
  sig: (0, 1, 6)  count: 1  vars: ['d0']

Class 7: size=1
  sig: (6, 8, 19)  count: 1  vars: ['d2']

Class 8: size=1
  sig: (10, 21, 22)  count: 1  vars: ['d7']

Class 9: size=1
  sig

In [24]:
from optimization.experiments_settings import from_edgelist, load_tanner_graph
from optimization.experiments_settings import codes, path_to_initial_codes, textfiles

C=2

path_to_initial_codes = "optimization/initial_codes/"
initial_state = load_tanner_graph(path_to_initial_codes + textfiles[C])

count_signature_equivalence_classes(initial_state)
    

Total variable cyclic-equivalence classes: 22/32
Total check cyclic-equivalence classes: 23/24

Check variable classes with cyclic-shift equivalence:
Class 1: size=8
  sig: (0, 1, 2)  count: 1  vars: ['d0']
  sig: (3, 4, 5)  count: 1  vars: ['d1']
  sig: (6, 7, 8)  count: 1  vars: ['d2']
  sig: (9, 10, 11)  count: 1  vars: ['d3']
  sig: (12, 13, 14)  count: 1  vars: ['d4']
  sig: (15, 16, 17)  count: 1  vars: ['d5']
  sig: (18, 19, 20)  count: 1  vars: ['d6']
  sig: (21, 22, 23)  count: 1  vars: ['d7']

Class 2: size=2
  sig: (1, 9, 12)  count: 1  vars: ['d9']
  sig: (5, 13, 16)  count: 1  vars: ['d12']

Class 3: size=2
  sig: (1, 7, 23)  count: 1  vars: ['d16']
  sig: (4, 6, 12)  count: 1  vars: ['d27']

Class 4: size=2
  sig: (16, 19, 23)  count: 1  vars: ['d23']
  sig: (1, 5, 22)  count: 1  vars: ['d24']

Class 5: size=1
  sig: (0, 3, 6)  count: 1  vars: ['d8']

Class 6: size=1
  sig: (2, 15, 18)  count: 1  vars: ['d10']

Class 7: size=1
  sig: (4, 10, 21)  count: 1  vars: ['d11']



In [8]:
# Compute and display distinct variable-neighborhood signatures (exact check indices)
check_nodes = sorted([n for n, b in state.nodes(data='bipartite') if b == 0])
var_nodes = sorted([n for n, b in state.nodes(data='bipartite') if b == 1])
check_index = {n: i for i, n in enumerate(check_nodes)}
var_index = {n: i for i, n in enumerate(var_nodes)}

patterns = {}
for v in var_nodes:
    nbrs = sorted([check_index[n] for n in state.neighbors(v)])
    sig = tuple(nbrs)
    patterns.setdefault(sig, []).append(v)

# Sort patterns by occurrence
sorted_patterns = sorted(patterns.items(), key=lambda kv: len(kv[1]), reverse=True)

print(f"Total variable nodes: {len(var_nodes)}")
print(f"Distinct variable-neighborhood signatures: {len(sorted_patterns)}\n")

for rank, (sig, vars_) in enumerate(sorted_patterns, start=1):
    local_vars = [f"d{var_index[v]}" for v in vars_[:10]]  # show up to 10 example variables
    checks = [f"c{i}" for i in sig]
    print(f"Pattern {rank}: count={len(vars_)}")
    print(f"  checks: {' '.join(checks)}")
    print(f"  example variables (up to 10): {' '.join(local_vars)}")
    print()

# Also show the two largest counts explicitly (for your example)
if len(sorted_patterns) >= 2:
    print("Top-2 pattern counts:")
    print(sorted_patterns[0][0], len(sorted_patterns[0][1]))
    print(sorted_patterns[1][0], len(sorted_patterns[1][1]))

Total variable nodes: 32
Distinct variable-neighborhood signatures: 32

Pattern 1: count=1
  checks: c0 c1 c6
  example variables (up to 10): d0

Pattern 2: count=1
  checks: c3 c4 c5
  example variables (up to 10): d1

Pattern 3: count=1
  checks: c6 c8 c19
  example variables (up to 10): d2

Pattern 4: count=1
  checks: c9 c10 c11
  example variables (up to 10): d3

Pattern 5: count=1
  checks: c12 c13 c14
  example variables (up to 10): d4

Pattern 6: count=1
  checks: c15 c16 c17
  example variables (up to 10): d5

Pattern 7: count=1
  checks: c18 c19 c20
  example variables (up to 10): d6

Pattern 8: count=1
  checks: c10 c21 c22
  example variables (up to 10): d7

Pattern 9: count=1
  checks: c0 c2 c3
  example variables (up to 10): d8

Pattern 10: count=1
  checks: c1 c8 c12
  example variables (up to 10): d9

Pattern 11: count=1
  checks: c2 c15 c18
  example variables (up to 10): d10

Pattern 12: count=1
  checks: c4 c10 c21
  example variables (up to 10): d11

Pattern 13: cou

In [15]:
# Test cyclic-shift equivalence of variable-neighborhood signatures
m = len(check_nodes)  # number of check nodes

def shift_sig(sig, s):
    return tuple(sorted(((i + s) % m) for i in sig))

# Build canonical representative for each signature under cyclic shifts
canonical_map = {}
for sig, vars_ in sorted_patterns:
    reps = [shift_sig(sig, s) for s in range(m)]
    canonical = min(reps)
    canonical_map.setdefault(canonical, []).append((sig, vars_))

print(f"Total cyclic-equivalence classes: {len(canonical_map)}")
print()

# Show classes that contain more than one distinct signature
for i, (canon, members) in enumerate(sorted(canonical_map.items(), key=lambda kv: -len(kv[1])), start=1):
    print(f"Class {i}: size={len(members)}")
    for sig, vars_ in members:
        print(f"  sig: {sig}  count: {len(vars_)}  vars: {[f'd{var_index[v]}' for v in vars_]}" )
    print()

Total cyclic-equivalence classes: 24

Class 1: size=5
  sig: (3, 4, 5)  count: 1  vars: ['d1']
  sig: (9, 10, 11)  count: 1  vars: ['d3']
  sig: (12, 13, 14)  count: 1  vars: ['d4']
  sig: (15, 16, 17)  count: 1  vars: ['d5']
  sig: (18, 19, 20)  count: 1  vars: ['d6']

Class 2: size=2
  sig: (2, 15, 18)  count: 1  vars: ['d10']
  sig: (2, 5, 13)  count: 1  vars: ['d26']

Class 3: size=2
  sig: (7, 8, 14)  count: 1  vars: ['d14']
  sig: (14, 15, 21)  count: 1  vars: ['d22']

Class 4: size=2
  sig: (1, 7, 23)  count: 1  vars: ['d16']
  sig: (4, 6, 12)  count: 1  vars: ['d27']

Class 5: size=2
  sig: (12, 17, 20)  count: 1  vars: ['d21']
  sig: (1, 17, 22)  count: 1  vars: ['d24']

Class 6: size=1
  sig: (0, 1, 6)  count: 1  vars: ['d0']

Class 7: size=1
  sig: (6, 8, 19)  count: 1  vars: ['d2']

Class 8: size=1
  sig: (10, 21, 22)  count: 1  vars: ['d7']

Class 9: size=1
  sig: (0, 2, 3)  count: 1  vars: ['d8']

Class 10: size=1
  sig: (1, 8, 12)  count: 1  vars: ['d9']

Class 11: size=

In [ ]:
classical_counts = count_classical_undetectable_errors(
    csr_H,
    distance=d_classical,
    offsets=range(0, 50),
    max_comb_order=CLASSICAL_MAX_COMB_ORDER,
)

print("Classical undetectable errors in ker(H):")
for weight, count in classical_counts.items():
    print(f"  weight {weight}: {count}")

Classical undetectable errors in ker(H):
  weight 12: 27
  weight 13: 0
  weight 14: 72
  weight 15: 0
  weight 16: 71
  weight 17: 0
  weight 18: 52
  weight 19: 0
  weight 20: 19
  weight 21: 0
  weight 22: 12
  weight 23: 0
  weight 24: 2
  weight 25: 0
  weight 26: 0
  weight 27: 0
  weight 28: 0
  weight 29: 0
  weight 30: 0
  weight 31: 0
  weight 32: 0
  weight 33: 0
  weight 34: 0
  weight 35: 0
  weight 36: 0
  weight 37: 0
  weight 38: 0
  weight 39: 0
  weight 40: 0
  weight 41: 0
  weight 42: 0
  weight 43: 0
  weight 44: 0
  weight 45: 0
  weight 46: 0
  weight 47: 0
  weight 48: 0
  weight 49: 0
  weight 50: 0
  weight 51: 0
  weight 52: 0
  weight 53: 0
  weight 54: 0
  weight 55: 0
  weight 56: 0
  weight 57: 0
  weight 58: 0
  weight 59: 0
  weight 60: 0
  weight 61: 0


In [7]:
css_counts = count_css_logical_basis_representatives(
    Hx,
    Hz,
    offsets=range(0, 50),
    distance=d_quantum,
    max_comb_order=CSS_MAX_COMB_ORDER,
)

scope = "exact full-span" if CSS_MAX_COMB_ORDER is None else f"basis combinations up to order {CSS_MAX_COMB_ORDER}"
print(f"CSS logical-basis representative counts ({scope}):")
for op_type, counts in css_counts.items():
    print(f"  {op_type}-type")
    for weight, count in counts.items():
        print(f"    weight {weight}: {count}")

CSS logical-basis representative counts (basis combinations up to order 5):
  X-type
    weight 12: 112
    weight 13: 0
    weight 14: 273
    weight 15: 0
    weight 16: 213
    weight 17: 0
    weight 18: 125
    weight 19: 0
    weight 20: 45
    weight 21: 0
    weight 22: 15
    weight 23: 0
    weight 24: 3229
    weight 25: 0
    weight 26: 13702
    weight 27: 0
    weight 28: 21600
    weight 29: 0
    weight 30: 17031
    weight 31: 0
    weight 32: 8638
    weight 33: 0
    weight 34: 3300
    weight 35: 0
    weight 36: 17027
    weight 37: 0
    weight 38: 108002
    weight 39: 0
    weight 40: 267224
    weight 41: 0
    weight 42: 321545
    weight 43: 0
    weight 44: 198140
    weight 45: 0
    weight 46: 79368
    weight 47: 0
    weight 48: 35938
    weight 49: 0
    weight 50: 168630
    weight 51: 0
    weight 52: 661855
    weight 53: 0
    weight 54: 1234594
    weight 55: 0
    weight 56: 1120491
    weight 57: 0
    weight 58: 523711
    weight 59: 0
    weigh

## Sampling random fixed-weight errors

This estimates the fraction of uniformly sampled weight-`w` errors with zero syndrome. For the CSS/HGP code, X errors are checked against `Hz`, and Z errors are checked against `Hx`.

In [ ]:
NUM_SAMPLES = 100000
SAMPLE_SEED = 12345
sample_weights = [int(d_quantum), int(d_quantum) + 1, int(d_quantum) + 2]

classical_sample_counts = sample_weights_undetectable_errors(
    csr_H,
    weights=sample_weights,
    num_samples=NUM_SAMPLES,
    seed=SAMPLE_SEED,
)

x_sample_counts = sample_weights_undetectable_errors(
    Hz,
    weights=sample_weights,
    num_samples=NUM_SAMPLES,
    seed=SAMPLE_SEED + 1,
)

z_sample_counts = sample_weights_undetectable_errors(
    Hx,
    weights=sample_weights,
    num_samples=NUM_SAMPLES,
    seed=SAMPLE_SEED + 2,
)

print(f"Random fixed-weight sampling ({NUM_SAMPLES} samples per weight):")
for weight in sample_weights:
    c = classical_sample_counts[weight]
    x = x_sample_counts[weight]
    z = z_sample_counts[weight]
    print(f"weight {weight}:")
    print(f"  classical ker(H): {c['undetectable_count']}/{NUM_SAMPLES} = {c['undetectable_fraction']:.6f}")
    print(f"  CSS X errors, ker(Hz): {x['undetectable_count']}/{NUM_SAMPLES} = {x['undetectable_fraction']:.6f}")
    print(f"  CSS Z errors, ker(Hx): {z['undetectable_count']}/{NUM_SAMPLES} = {z['undetectable_fraction']:.6f}")

Random fixed-weight sampling (100000 samples per weight):
weight 12:
  classical ker(H): 0/100000 = 0.000000
  CSS X errors, ker(Hz): 0/100000 = 0.000000
  CSS Z errors, ker(Hx): 0/100000 = 0.000000
weight 13:
  classical ker(H): 0/100000 = 0.000000
  CSS X errors, ker(Hz): 0/100000 = 0.000000
  CSS Z errors, ker(Hx): 0/100000 = 0.000000
weight 14:
  classical ker(H): 0/100000 = 0.000000
  CSS X errors, ker(Hz): 0/100000 = 0.000000
  CSS Z errors, ker(Hx): 0/100000 = 0.000000
